In [113]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [114]:
df = pd.read_csv(r"C:\Users\Asus\Desktop\Laptop Price\laptop_price.csv")

In [115]:
df.head(3)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,Intel HD Graphics 620,No OS,1.86kg,575.00


In [116]:
df.isnull().sum()

laptop_ID           0
Company             0
Product             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Gpu                 0
OpSys               0
Weight              0
Price_euros         0
dtype: int64

In [117]:
df['Ram'] = df['Ram'].astype(str).str.replace('GB', '', regex=False).astype(int)
df['Weight'] = df['Weight'].astype(str).str.replace('kg', '', regex=False).astype(float)

In [118]:
df.head(3)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,Intel Iris Plus Graphics 640,macOS,1.37,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,Intel HD Graphics 6000,macOS,1.34,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,Intel HD Graphics 620,No OS,1.86,575.00


In [119]:
df = df.drop(columns=['laptop_ID','Inches','Product','ScreenResolution','Cpu','Gpu'])

In [120]:
df.head(2)

,Company,TypeName,Ram,OpSys,Weight,Price_euros
0,Apple,Ultrabook,8,macOS,1.37,1339.69
1,Apple,Ultrabook,8,macOS,1.34,898.94


In [121]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [122]:
X = df.drop(columns=['Price_euros'])  
y = df['Price_euros']                  

In [123]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1303, 5)
Target shape: (1303,)


In [124]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [125]:
categorical_columns = [
    "Company",
    "TypeName",
    "OpSys"]

numerical_columns = [
    "Ram",
    "Weight"]

In [126]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        )
    ],
    remainder="passthrough")

In [127]:
model = RandomForestRegressor(n_estimators=100,random_state=42)

In [128]:
pipeline = Pipeline(
    steps=[("preprocessor", preprocessor),("model", model)]
)

In [129]:
pipeline.fit(X_train,y_train)
print("Model training completed!")

Model training completed!


In [130]:
y_pred = pipeline.predict(X_test)
print(y_pred[:10])

[1454.345      1569.04635714  611.79063095  902.07981667 1484.28009524
 2070.21820952  760.59905159  557.87223377 1225.95513333  544.617386  ]


In [131]:
mae = mean_absolute_error(y_test,y_pred)
print(f"Mean Absolute Error: {mae:.2f} euros")

Mean Absolute Error: 220.27 euros


In [132]:
r2 = r2_score(y_test,y_pred)
print(f"R² Score: {r2:.2f}")

R² Score: 0.76


In [133]:
new_laptop = pd.DataFrame({
    "Company": ["HP"],
    "TypeName": ["Ultrabook"],
    "Ram": [8],
    "OpSys": ["Windows 10"],
    "Weight": [1.8]
})

print(new_laptop)

  Company   TypeName  Ram       OpSys  Weight
0      HP  Ultrabook    8  Windows 10     1.8


In [135]:
predicted_price = pipeline.predict(new_laptop)
print(f"Predicted Price: €{predicted_price[0]:.2f}")

Predicted Price: €1110.31
